# Kvasir-SEG U-Net Baseline (ResNet34 name-compatible)

Runs a segmentation baseline and writes standardized artifacts.\n\nIf `segmentation_models_pytorch` is installed, uses Unet-ResNet34.\nOtherwise falls back to an internal UNet implementation.

In [1]:
import sys
from pathlib import Path

def _bootstrap_kvasir_seg_path() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / 'utils' / 'segmentation_common.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
        alt = p / 'Prototyping_reformat' / 'DatasetAnalysis' / 'Kvasir_SEG'
        if (alt / 'utils' / 'segmentation_common.py').exists():
            if str(alt) not in sys.path:
                sys.path.insert(0, str(alt))
            return alt
    raise RuntimeError('Could not locate Kvasir_SEG utils path from current working directory.')

BOOTSTRAP_ROOT = _bootstrap_kvasir_seg_path()
print('BOOTSTRAP_ROOT:', BOOTSTRAP_ROOT)

BOOTSTRAP_ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG


In [2]:

import os
import json
from pathlib import Path

import pandas as pd
import torch

from utils.segmentation_common import (
    find_kvasir_seg_root,
    load_metadata,
    TrainConfig,
    UNetSmall,
    train_and_evaluate,
)

ROOT = find_kvasir_seg_root()
META_CSV = ROOT / '0_dataset_prep' / 'out' / 'metadata' / 'metadata_enriched.csv'
SPLIT_HASH_TXT = ROOT / '0_dataset_prep' / 'out' / 'metadata' / 'split_hash.txt'
OUT_DIR = ROOT / '1_classic_seg_baselines' / 'out' / 'unet_resnet34_baseline'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.getenv('SEED', '42'))
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '8'))
NUM_WORKERS = int(os.getenv('NUM_WORKERS', '2'))
EPOCHS = int(os.getenv('EPOCHS', '8'))
LR = float(os.getenv('LR', '1e-3'))
WEIGHT_DECAY = float(os.getenv('WEIGHT_DECAY', '1e-4'))
IMAGE_SIZE = int(os.getenv('IMAGE_SIZE', '352'))
THRESHOLD = float(os.getenv('THRESHOLD', '0.5'))

MAX_TRAIN = int(os.getenv('MAX_TRAIN_SAMPLES', '0')) or None
MAX_VAL = int(os.getenv('MAX_VAL_SAMPLES', '0')) or None
MAX_TEST = int(os.getenv('MAX_TEST_SAMPLES', '0')) or None

print('ROOT:', ROOT)
print('OUT_DIR:', OUT_DIR)
print('CUDA available:', torch.cuda.is_available())


ROOT: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG
OUT_DIR: /mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/1_classic_seg_baselines/out/unet_resnet34_baseline
CUDA available: True


In [3]:

meta_df = load_metadata(META_CSV)
split_hash = SPLIT_HASH_TXT.read_text().strip() if SPLIT_HASH_TXT.exists() else None

cfg = TrainConfig(
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    threshold=THRESHOLD,
    compute_hd95=False,
    save_pred_masks=True,
    pred_mask_limit=200,
)

model_name = 'unet_small'
model = UNetSmall(in_ch=3, out_ch=1, base=32)

# Optional drop-in to match filename semantics when dependency exists
try:
    import segmentation_models_pytorch as smp  # type: ignore
    model = smp.Unet(encoder_name='resnet34', encoder_weights=None, classes=1, in_channels=3)
    model_name = 'unet_resnet34_smp'
    print('Using segmentation_models_pytorch Unet-ResNet34')
except Exception as e:
    print('segmentation_models_pytorch not available, using UNetSmall fallback:', e)


segmentation_models_pytorch not available, using UNetSmall fallback: No module named 'segmentation_models_pytorch'


In [4]:

results = train_and_evaluate(
    model=model,
    model_name=model_name,
    root=ROOT,
    out_dir=OUT_DIR,
    cfg=cfg,
    metadata_df=meta_df,
    split_hash=split_hash,
    max_train=MAX_TRAIN,
    max_val=MAX_VAL,
    max_test=MAX_TEST,
)

print(json.dumps(results, indent=2))


Epoch 1/8 train_loss=0.6059 val_loss=0.5619 val_dice=0.4264
Epoch 2/8 train_loss=0.5153 val_loss=0.7143 val_dice=0.4349
Epoch 3/8 train_loss=0.4765 val_loss=0.4851 val_dice=0.4889
Epoch 4/8 train_loss=0.4527 val_loss=0.4473 val_dice=0.4683
Epoch 5/8 train_loss=0.4297 val_loss=0.4703 val_dice=0.4455
Epoch 6/8 train_loss=0.4176 val_loss=0.4660 val_dice=0.5219
Epoch 7/8 train_loss=0.4114 val_loss=0.4400 val_dice=0.5168
Epoch 8/8 train_loss=0.4087 val_loss=0.4196 val_dice=0.5381


/mnt/hf/thesis/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_SEG/utils/segmentation_common.py:1390: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_st

{
  "train": {
    "n": 800,
    "dice_mean": 0.5615835207760003,
    "dice_median": 0.6011665481516759,
    "dice_std": 0.23726847546410013,
    "iou_mean": 0.42657210468938245,
    "iou_median": 0.4297629452675191,
    "iou_std": 0.22237939664318185,
    "precision_mean": 0.5533811761011053,
    "precision_median": 0.5555835098475184,
    "precision_std": 0.2916802531117133,
    "recall_mean": 0.7128536857295168,
    "recall_median": 0.789986332123423,
    "recall_std": 0.26787654245310477,
    "f1_mean": 0.5615835165569868,
    "f1_median": 0.6011665435510983,
    "f1_std": 0.23726847461239203,
    "specificity_mean": 0.9061585664640875,
    "specificity_median": 0.93389687875186,
    "specificity_std": 0.0878924776484423,
    "loss": 0.4054561641812324
  },
  "val": {
    "n": 100,
    "dice_mean": 0.5380757756501058,
    "dice_median": 0.5763373296918965,
    "dice_std": 0.2558300316096999,
    "iou_mean": 0.40961661310852393,
    "iou_median": 0.4048271965836038,
    "iou_std": 0